# Method of Lines and Runge-Kutta

This notebook connects the finite-difference right-hand side of an evolution system to Runge-Kutta time integration. It uses NRPy's current Butcher-table utilities to inspect RK4 and to show the gridfunction storage names used by generated Method of Lines code.

[Index](../index.ipynb) | Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) | Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)

## Why This Matters

The Method of Lines separates space and time: finite differences handle spatial derivatives, and Runge-Kutta stages advance the fields in time.

## What You Will See

- RK4 table and stage-storage names.
- Generated Method-of-Lines registration output.
- Discovery of the Cartesian wave generator.

## Table of Contents

1. [Method of Lines split](#Method-of-Lines-split)
2. [NRPy RK table utilities](#NRPy-RK-table-utilities)
3. [RK4 table and stage storage](#RK4-table-and-stage-storage)
4. [Generated MoL driver registration](#Generated-MoL-driver-registration)
5. [Generator discoverability](#Generator-discoverability)
6. [Next notebooks](#Next-notebooks)

## Method of Lines Split

After spatial discretization, an evolution equation has the form

$$
\frac{d y}{dt} = F(t,y).
$$

The right-hand side $F$ contains all finite-difference derivatives and boundary-sensitive data access. A Runge-Kutta method advances the semi-discrete state by evaluating this right-hand side at intermediate stages and combining the stage derivatives with weights from a Butcher table.

## NRPy RK Table Utilities

NRPy stores supported Runge-Kutta methods as Butcher tables. The functions used here validate the table, report whether it is diagonal, and return the intermediate gridfunction names needed by the generated time-stepper.

In [1]:
import importlib.util

import nrpy
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import (
    register_CFunction_MoL_step_forward_in_time,
)
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)


print("nrpy import succeeded")

butcher_tables = generate_Butcher_tables()
validate(butcher_tables, ["RK4"], verbose=False)
print("RK4 table is available and validated.")

nrpy import succeeded


RK4: PASSED VALIDATION
RK4 table is available and validated.


## RK4 Table and Stage Storage

Classical RK4 has four right-hand-side evaluations and fourth-order time accuracy. The stage storage names below are the gridfunction arrays that NRPy uses in generated BHaH infrastructure code.

In [2]:
rk4_table, rk4_order = butcher_tables["RK4"]

print("RK4 order:", rk4_order)
print("RK4 is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print("RK4 intermediate gridfunctions:")
for name in intermediate_stage_gf_names_list(butcher_tables, "RK4"):
    print(" ", name)

print("\nRK4 Butcher table rows:")
for row in rk4_table:
    print(" ", row)

RK4 order: 4
RK4 is diagonal: True
RK4 intermediate gridfunctions:
  y_nplus1_running_total_gfs
  k_odd_gfs
  k_even_gfs

RK4 Butcher table rows:
  [0]
  [1/2, 1/2]
  [1/2, 0, 1/2]
  [1, 0, 0, 1]
  ['', 1/6, 1/3, 1/3, 1/6]


## Generated MoL Driver Registration

The full generated driver is registered as a C function. The short placeholder calls below stand in for a project-specific right-hand-side kernel and a post-right-hand-side operation such as a boundary fill. The cell prints a small excerpt of the registered function when formatting support is available in the current Python session.

In [3]:
import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH


rhs_call = "rhs_eval(commondata, params, xx, auxevol_gfs, RK_INPUT_GFS, RK_OUTPUT_GFS);"
post_rhs_call = "apply_bcs(commondata, params, RK_OUTPUT_GFS);"

mol_demo_c_function_name = "MoL_step_forward_in_time"
mol_demo_substep_names = [f"RK_STEP{i}" for i in range(1, 5)]

previous_mol_c_function = cfc.CFunction_dict.pop(mol_demo_c_function_name, None)
previous_mol_substeps = {}
for substep_name in mol_demo_substep_names:
    if substep_name in BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict:
        previous_mol_substeps[substep_name] = (
            BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(substep_name)
        )

try:
    register_CFunction_MoL_step_forward_in_time(
        butcher_tables,
        "RK4",
        rhs_string=rhs_call,
        post_rhs_string=post_rhs_call,
    )
except Exception as exc:
    print(f"MoL registration demonstration did not complete: {type(exc).__name__}: {exc}")
else:
    mol_function = cfc.CFunction_dict["MoL_step_forward_in_time"].full_function
    print("Registered C function:", "MoL_step_forward_in_time")
    print("\n".join(mol_function.splitlines()[:18]))
finally:
    cfc.CFunction_dict.pop(mol_demo_c_function_name, None)
    if previous_mol_c_function is not None:
        cfc.CFunction_dict[mol_demo_c_function_name] = previous_mol_c_function
    for substep_name in mol_demo_substep_names:
        BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(substep_name, None)
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.update(previous_mol_substeps)

Registered C function: MoL_step_forward_in_time
#include "BHaH_defines.h"
#include "BHaH_function_prototypes.h"

#define LOOP_ALL_GFS_GPS(ii)                                                                                                                         \
  _Pragma("omp parallel for simd") for (int(ii) = 0;                                                                                                 \
                                        (ii) < params->Nxx_plus_2NGHOSTS0 * params->Nxx_plus_2NGHOSTS1 * params->Nxx_plus_2NGHOSTS2 * NUM_EVOL_GFS;  \
                                        (ii)++)

/**
 * Kernel: rk_substep_1_host.
 * Compute RK substep 1.
 */
static void rk_substep_1_host(params_struct *restrict params, REAL *restrict k_odd_gfs, REAL *restrict y_n_gfs,
                              REAL *restrict y_nplus1_running_total_gfs, const REAL dt) {
  LOOP_ALL_GFS_GPS(i) {
    const REAL k_odd_gfsL = k_odd_gfs[i];
    const REAL y_n_gfsL = y_n_gfs[i];
    const REAL R

## Generator Discoverability

Generated-project notebooks should check whether the Cartesian wave-equation generator exists without importing it at top level. Importing the generator performs registration work, so discovery is enough here.

In [4]:
generator_name = "nrpy.examples.wave_equation_cartesian"
generator_spec = importlib.util.find_spec(generator_name)
if generator_spec is None:
    print(f"Generator not found: {generator_name}")
else:
    print(f"Generator discovered: {generator_name}")

Generator discovered: nrpy.examples.wave_equation_cartesian


## Next Notebooks

[Boundary conditions and convergence](boundary_conditions_and_convergence.ipynb) explains ghost-zone fills and convergence reruns. [Wave equation and C code generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb) then constructs the symbolic wave-equation right-hand side used by the generated Cartesian project.

## Where This Leads

- [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) reviews the prerequisite step.
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.